# Análise de K-means 1D (OpenMP) — Etapa 1

Este notebook carrega os resultados CSV (e.g., `results104.csv`, `results105.csv`, `results106.csv` **ou** saídas do `run_omp_sweep_v2.sh`), normaliza colunas derivadas e gera gráficos de **tempo**, **speedup** e **throughput** por número de threads, além de comparar `schedule` e `chunk`.

> **Speedup** é calculado como `tempo_baseline_t1 / tempo(T)`, onde `tempo_baseline_t1` é o **menor tempo** observado com `T=1` (baseline único por (N,K)).
> **Throughput** é `N / (ms/1000)` (pontos por segundo).


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Liste aqui os arquivos que quer analisar
arquivos = [
    Path("/mnt/data/results104.csv"),
    Path("/mnt/data/results105.csv"),
    Path("/mnt/data/results106.csv"),
    # Adicione também qualquer CSV gerado pelo run_omp_sweep_v2.sh (por exemplo, Path("/mnt/data/results_v2.csv"))
]

dfs = []
for p in arquivos:
    if p.exists():
        try:
            df = pd.read_csv(p)
        except Exception:
            # Tenta detectar se não há cabeçalho e as colunas estão na ordem do script v1:
            # T,schedule,chunk,N,K,max_iter,eps,iterations,ms,sse,monotonic,speedup
            # Se falhar, tenta leitura sem header para o formato do binário (N,K,max_iter,eps,T,schedule,chunk,iterations,ms,sse,monotonic)
            df = pd.read_csv(p, header=None)
            if df.shape[1] == 12:
                df.columns = ["T","schedule","chunk","N","K","max_iter","eps","iterations","ms","sse","monotonic","speedup"]
            elif df.shape[1] == 11:
                df.columns = ["N","K","max_iter","eps","T","schedule","chunk","iterations","ms","sse","monotonic"]
        dfs.append(df)
    else:
        print(f"Aviso: arquivo não encontrado: {p}")
        
assert dfs, "Nenhum CSV encontrado."

df = pd.concat(dfs, ignore_index=True)

# Coerção de tipos
for col in ["T","chunk","N","K","max_iter","iterations","monotonic"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

for col in ["ms","sse","eps","speedup"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Se vier no formato do binário, reordene + derive colunas para compatibilidade
if {"N","K","max_iter","eps","T","schedule","chunk","iterations","ms","sse","monotonic"}.issubset(df.columns) and "speedup" not in df.columns:
    df["speedup"] = pd.NA

# Throughput
if "ms" in df.columns and "N" in df.columns:
    df["points_per_sec"] = (df["N"] * 1000.0) / df["ms"]

# Recalcular speedup com baseline único (em caso de CSVs antigos)
def recompute_speedup(group):
    # baseline: menor ms em T=1 dentro do grupo
    base = group.loc[group["T"] == 1, "ms"].min()
    if pd.isna(base):
        return group  # nada a fazer
    group = group.copy()
    group["speedup_calc"] = base / group["ms"]
    # se já existir 'speedup', mantenha como coluna, mas privilegie 'speedup_calc' para gráficos
    return group

df = df.groupby(["N","K"], dropna=False, group_keys=False).apply(recompute_speedup)

# Salva uma versão normalizada para referência
out_norm = Path("/mnt/data/results_normalized.csv")
df.to_csv(out_norm, index=False)
print(f"Arquivo normalizado salvo em: {out_norm}")
df.head()


## Tempo (ms) vs Threads — por (N,K) e `schedule`

In [ ]:

# Um gráfico por par (N,K)
nk_groups = df.groupby(["N","K"])
for (N,K), g in nk_groups:
    plt.figure()
    g2 = g.sort_values(["schedule","T"])
    for sched, gs in g2.groupby("schedule"):
        xs = gs["T"]
        ys = gs["ms"]
        plt.plot(xs, ys, marker="o", label=str(sched))
    plt.title(f"Tempo vs Threads (N={N}, K={K})")
    plt.xlabel("Threads (T)")
    plt.ylabel("Tempo (ms)")
    plt.legend()
    plt.grid(True)
    plt.show()


## Speedup vs Threads — baseline único T=1

In [ ]:

for (N,K), g in df.groupby(["N","K"]):
    plt.figure()
    g2 = g.sort_values(["schedule","T"])
    for sched, gs in g2.groupby("schedule"):
        xs = gs["T"]
        ys = gs.get("speedup_calc", gs.get("speedup"))
        plt.plot(xs, ys, marker="o", label=str(sched))
    plt.title(f"Speedup vs Threads (baseline T=1) — N={N}, K={K}")
    plt.xlabel("Threads (T)")
    plt.ylabel("Speedup")
    plt.legend()
    plt.grid(True)
    plt.show()


## Throughput (pontos/s) vs Threads

In [ ]:

for (N,K), g in df.groupby(["N","K"]):
    plt.figure()
    g2 = g.sort_values(["schedule","T"])
    for sched, gs in g2.groupby("schedule"):
        xs = gs["T"]
        ys = gs["points_per_sec"]
        plt.plot(xs, ys, marker="o", label=str(sched))
    plt.title(f"Throughput vs Threads (N={N}, K={K})")
    plt.xlabel("Threads (T)")
    plt.ylabel("Pontos por segundo")
    plt.legend()
    plt.grid(True)
    plt.show()


## Efeito do `chunk` (fixando `schedule`)

In [ ]:

# Analisa chunk separadamente por schedule e T
for (N,K), g in df.groupby(["N","K"]):
    for sched, gs in g.groupby("schedule"):
        for T, gt in gs.groupby("T"):
            if gt["chunk"].nunique() > 1:
                plt.figure()
                xx = gt["chunk"]
                yy = gt["ms"]
                plt.plot(xx, yy, marker="o")
                plt.title(f"Tempo vs Chunk (N={N}, K={K}, schedule={sched}, T={T})")
                plt.xlabel("chunk")
                plt.ylabel("Tempo (ms)")
                plt.grid(True)
                plt.show()
